# Lab 4 — Technique Toolbox

**Module:** Mechanism-Based Prompting Techniques | **Duration:** 90 min | **Pair programming:** Switch at 55 min

**Fil Rouge:** DevAssist — AI-Powered Developer Documentation Assistant for TaskFlow

---

## Objectives

1. Build a 3-phase structured extraction pipeline (free-form → format → few-shot)
2. Measure compliance rates with jsonschema validation at each phase
3. Run a self-consistency mini-bench and chart accuracy vs. cost
4. Connect each technique to its underlying mechanism

**Technique → Mechanism Map:**

| Technique | Mechanism | Module |
|-----------|-----------|--------|
| Few-shot | Attention over demonstrated patterns | M2 |
| Chain-of-thought | Intermediate tokens as new context | M2 |
| Self-consistency | Sampling diversity + majority voting | M1 |
| Format constraints | Structural token patterns + validation | M2 |

---
## Setup

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '.')

from utils.generation_utils import generate, generate_n, is_ollama_available, try_parse_json
from utils.schema_validators import validate_user_story_output, compute_compliance_rate, compliance_summary
from utils.consistency_utils import extract_final_answer, majority_vote, compute_agreement_rate

print('✓ All imports successful!')

In [ ]:
OLLAMA_AVAILABLE = is_ollama_available()
print(f"Ollama: {'✓ Running' if OLLAMA_AVAILABLE else '⚠ Unavailable — using precomputed'}")

with open('data/precomputed_outputs.json', 'r') as f:
    PRECOMPUTED = json.load(f)
print('✓ Precomputed outputs loaded.')

In [ ]:
with open('data/user_stories.json', 'r') as f:
    stories_data = json.load(f)
user_stories = stories_data['stories']
print(f'Loaded {len(user_stories)} user stories:')
for s in user_stories:
    print(f"  [{s.get('difficulty','?'):6s}] #{s['id']:2d}: {s['text'][:70]}...")

In [ ]:
with open('data/logic_problems.json', 'r') as f:
    problems_data = json.load(f)
logic_problems = problems_data['problems']
print(f'Loaded {len(logic_problems)} logic/arithmetic problems.')

---
# PART 1 — STRUCTURED OUTPUT SPRINT (55 minutes)

**Task:** Extract `{actor, action, goal}` from TaskFlow user stories.

**🟢 Driver starts here.**

---
## Phase 1: Free-Form Extraction (15 min)

Write a simple extraction prompt. Do **NOT** specify the output format.

In [ ]:
# PHASE 1: FREE-FORM EXTRACTION (zero-shot, no format spec)

freeform_prompt = """Extract the actor, action, and goal from the following user story:

{user_story}"""
# TODO: Modify if you want, but keep it simple — no format spec!

print(f'Phase 1 prompt:\n{freeform_prompt}')

In [ ]:
results_p1 = []
if OLLAMA_AVAILABLE:
    for s in user_stories:
        prompt = freeform_prompt.format(user_story=s['text'])
        response = generate(prompt)
        validation = validate_user_story_output(response)
        results_p1.append({'id': s['id'], 'response': response, 'valid': validation['valid'], 'error': validation.get('error')})
        print(f"  {'✓' if validation['valid'] else '✗'} Story #{s['id']}: {response[:120]}...")
else:
    print('Using precomputed Phase 1 outputs:')
    for i, out in enumerate(PRECOMPUTED['extraction_sprint']['phase1_freeform']['sample_outputs']):
        v = validate_user_story_output(out)
        results_p1.append({'id': i+1, 'response': out, 'valid': v['valid'], 'error': v.get('error')})
        print(f"  {'✓' if v['valid'] else '✗'} Story #{i+1}: {out[:120]}...")

rate_p1 = compute_compliance_rate(results_p1)
print(f'\nPhase 1 compliance rate: {rate_p1:.0%}')
print(compliance_summary(results_p1))

### Phase 1 Analysis

**Q1: How many different output formats did the model use?**

TODO

**Q2: Could you write a single parser for all outputs?**

TODO

**Q3: What does this tell you about free-form extraction in production?**

TODO

---
## Phase 2: Format-Constrained Extraction (15 min)

Add explicit JSON format constraints. Validate with jsonschema.

In [ ]:
# PHASE 2: FORMAT-CONSTRAINED EXTRACTION

format_prompt = """Extract the actor, action, and goal from the user story below.

Return ONLY a valid JSON object with exactly these three string fields:
- "actor": the person or role performing the action
- "action": what they want to do
- "goal": why they want to do it (empty string "" if not stated)

No additional text, no markdown fences, no explanation. Just the JSON object.

User story: {user_story}"""
# TODO: Refine if you want

print(f'Phase 2 prompt:\n{format_prompt}')

In [ ]:
results_p2 = []
if OLLAMA_AVAILABLE:
    for s in user_stories:
        prompt = format_prompt.format(user_story=s['text'])
        response = generate(prompt)
        validation = validate_user_story_output(response)
        results_p2.append({'id': s['id'], 'response': response, 'valid': validation['valid'], 'error': validation.get('error')})
        print(f"  {'✓' if validation['valid'] else '✗'} Story #{s['id']}: {response[:120]}")
else:
    print('Using precomputed Phase 2 outputs:')
    for i, out in enumerate(PRECOMPUTED['extraction_sprint']['phase2_format']['sample_outputs']):
        v = validate_user_story_output(out)
        results_p2.append({'id': i+1, 'response': out, 'valid': v['valid'], 'error': v.get('error')})
        print(f"  {'✓' if v['valid'] else '✗'} Story #{i+1}: {out[:120]}")

rate_p2 = compute_compliance_rate(results_p2)
print(f'\nPhase 2 compliance rate: {rate_p2:.0%}')
print(compliance_summary(results_p2))

### Phase 2 Analysis

**Q1: How does compliance compare to Phase 1?**

TODO

**Q2: What types of errors remain?**

TODO

**Q3: Mechanistic explanation — why did format constraints improve results?**

TODO

---
## Phase 3: Few-Shot + Format-Constrained Extraction (15 min)

Add 2–3 examples. Choose wisely: cover different difficulties.

In [ ]:
# PHASE 3: FEW-SHOT + FORMAT-CONSTRAINED

fewshot_prompt = """Extract the actor, action, and goal from user stories.
Return ONLY a valid JSON object with fields: actor, action, goal.
If the goal is not stated, use an empty string.

Example 1:
User story: "As a developer, I want to search code by function name so that I can find relevant implementations quickly."
Output: {"actor": "developer", "action": "search code by function name", "goal": "find relevant implementations quickly"}

Example 2:
User story: "We need dark mode for the dashboard."
Output: {"actor": "user", "action": "enable dark mode for the dashboard", "goal": ""}

Now extract:
User story: "{user_story}"
Output:"""
# TODO: Modify examples if you want

print(f'Phase 3 prompt (first 300 chars):\n{fewshot_prompt[:300]}...')

In [ ]:
results_p3 = []
if OLLAMA_AVAILABLE:
    for s in user_stories:
        prompt = fewshot_prompt.format(user_story=s['text'])
        response = generate(prompt)
        validation = validate_user_story_output(response)
        results_p3.append({'id': s['id'], 'response': response, 'valid': validation['valid'], 'error': validation.get('error')})
        print(f"  {'✓' if validation['valid'] else '✗'} Story #{s['id']}: {response[:120]}")
else:
    print('Using precomputed Phase 3 outputs:')
    for i, out in enumerate(PRECOMPUTED['extraction_sprint']['phase3_fewshot']['sample_outputs']):
        v = validate_user_story_output(out)
        results_p3.append({'id': i+1, 'response': out, 'valid': v['valid'], 'error': v.get('error')})
        print(f"  {'✓' if v['valid'] else '✗'} Story #{i+1}: {out[:120]}")

rate_p3 = compute_compliance_rate(results_p3)
print(f'\nPhase 3 compliance rate: {rate_p3:.0%}')
print(compliance_summary(results_p3))

### Phase 3 Analysis

**Q1: What is the compliance rate progression?**

TODO

**Q2: Did few-shot improve extraction ACCURACY (not just format)?**

TODO

**Q3: For remaining failures — could different examples fix them?**

TODO

---
## Compliance Rate Comparison

In [ ]:
phases = ['Phase 1\n(Free-form)', 'Phase 2\n(Format)', 'Phase 3\n(Few-shot)']
rates = [rate_p1, rate_p2, rate_p3]

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c', '#f39c12', '#27ae60']
bars = ax.bar(phases, [r * 100 for r in rates], color=colors, width=0.5, edgecolor='white')
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{rate:.0%}', ha='center', va='bottom', fontsize=14, fontweight='bold')
ax.set_ylabel('JSON Schema Compliance (%)', fontsize=12)
ax.set_title('Structured Output Sprint — Compliance Progression', fontsize=14)
ax.set_ylim(0, 105)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### Compliance Comparison Table

| Phase | Compliance Rate | Common Errors |
|-------|----------------|---------------|
| Free-form | TODO% | TODO |
| Format-only | TODO% | TODO |
| Few-shot | TODO% | TODO |

### Mechanistic Summary

TODO: Write 2–3 sentences explaining why each phase improved, using M1–M4 concepts.

---
## Bonus: CoT for Ambiguous Stories (10 min, if time permits)

In [ ]:
cot_extraction_prompt = """Extract the actor, action, and goal from this user story.

First, analyze the story:
- Who is the actor? If not explicitly stated, infer the most likely role.
- What is the action? If multiple actions, combine them.
- What is the goal? If not stated, use empty string.

Then output ONLY a JSON object: {{"actor": "...", "action": "...", "goal": "..."}}

User story: "{user_story}"

Analysis:"""

ambiguous_ids = [11, 12, 14]
ambiguous_stories = [s for s in user_stories if s['id'] in ambiguous_ids]

if OLLAMA_AVAILABLE:
    for s in ambiguous_stories:
        print(f"\n--- Story #{s['id']}: {s['text'][:60]}... ---")
        resp_direct = generate(fewshot_prompt.format(user_story=s['text']))
        v_direct = validate_user_story_output(resp_direct)
        resp_cot = generate(cot_extraction_prompt.format(user_story=s['text']), max_tokens=600)
        v_cot = validate_user_story_output(resp_cot)
        print(f"  Direct: {'✓' if v_direct['valid'] else '✗'} → {resp_direct[:100]}")
        print(f"  CoT:    {'✓' if v_cot['valid'] else '✗'} → {resp_cot[:200]}")
else:
    print('⚠ Ollama unavailable. Analyze: would CoT help for extraction?')
    print('   Hint: extraction is pattern-matching, not multi-step reasoning.')

### CoT Analysis

**Did CoT help for ambiguous stories?** TODO

**Is CoT worth the extra token cost for extraction?** TODO

---
# PART 2 — SELF-CONSISTENCY MINI-BENCH (25 minutes)

**🔄 Switch driver/navigator.**

---
## Step 1: Single-Attempt Baseline

In [ ]:
cot_prompt = """Solve this problem step by step.
After your reasoning, write your final answer on a new line as:
ANSWER: <your answer>

Problem: {problem}
Solution: Let's think step by step."""

print('Running single-attempt baseline (n=1)...\n')
baseline_results = []

if OLLAMA_AVAILABLE:
    for p in logic_problems[:10]:
        response = generate(cot_prompt.format(problem=p['text']), temperature=0.7)
        answer = extract_final_answer(response)
        correct = (str(answer) == str(p['correct_answer']))
        baseline_results.append({'id': p['id'], 'answer': answer, 'correct_answer': p['correct_answer'], 'correct': correct})
        print(f"  {'✓' if correct else '✗'} #{p['id']}: got '{answer}', expected '{p['correct_answer']}'")
else:
    print('Using precomputed baseline accuracy.')
    single_acc = PRECOMPUTED['self_consistency']['n_1_accuracy']
    print(f'  Precomputed single-attempt accuracy: {single_acc:.0%}')
    for i, p in enumerate(logic_problems[:10]):
        baseline_results.append({'id': p['id'], 'correct': i < int(single_acc * 10)})

baseline_accuracy = sum(1 for r in baseline_results if r['correct']) / len(baseline_results)
print(f'\nBaseline accuracy (n=1): {baseline_accuracy:.0%}')

---
## Step 2: Self-Consistency with Majority Voting

In [ ]:
N_SAMPLES = 5
TEMPERATURE = 0.7
print(f'Running self-consistency (n={N_SAMPLES}, T={TEMPERATURE})...\n')

sc_results = []
if OLLAMA_AVAILABLE:
    for p in logic_problems[:10]:
        answers = []
        for _ in range(N_SAMPLES):
            resp = generate(cot_prompt.format(problem=p['text']), temperature=TEMPERATURE)
            answers.append(extract_final_answer(resp))
        consensus = majority_vote(answers)
        correct = (str(consensus) == str(p['correct_answer']))
        agreement = compute_agreement_rate(answers)
        sc_results.append({'id': p['id'], 'answers': answers, 'consensus': consensus, 'correct': correct, 'agreement': agreement})
        print(f"  {'✓' if correct else '✗'} #{p['id']}: answers={answers} → '{consensus}' (agr={agreement:.0%})")
else:
    print('Using precomputed self-consistency results.')
    sc_acc = PRECOMPUTED['self_consistency']['n_5_accuracy']
    print(f'  Precomputed n=5 accuracy: {sc_acc:.0%}')
    for i, p in enumerate(logic_problems[:10]):
        sc_results.append({'id': p['id'], 'correct': i < int(sc_acc * 10), 'agreement': 0.8})

sc_accuracy = sum(1 for r in sc_results if r['correct']) / len(sc_results)
print(f'\nSelf-consistency accuracy (n={N_SAMPLES}): {sc_accuracy:.0%}')
print(f'Improvement over baseline: +{(sc_accuracy - baseline_accuracy):.0%}')
print(f'Cost multiplier: {N_SAMPLES}×')

---
## Step 3: Gain-vs-Cost Chart

In [ ]:
if OLLAMA_AVAILABLE:
    n_values = [1, N_SAMPLES]
    accuracies = [baseline_accuracy * 100, sc_accuracy * 100]
    print('TODO: Extend to more n values if time permits.')
else:
    sc_data = PRECOMPUTED['self_consistency']
    n_values = [1, 3, 5, 7]
    accuracies = [sc_data['n_1_accuracy']*100, sc_data['n_3_accuracy']*100, sc_data['n_5_accuracy']*100, sc_data['n_7_accuracy']*100]

fig, ax1 = plt.subplots(figsize=(9, 5))
color_acc = '#2980b9'
ax1.plot(n_values, accuracies, 'o-', color=color_acc, linewidth=2.5, markersize=10, label='Accuracy')
ax1.set_xlabel('Number of Samples (n)', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12, color=color_acc)
ax1.tick_params(axis='y', labelcolor=color_acc)
ax1.set_ylim(0, 100)
ax1.set_xticks(n_values)
ax1.axhline(y=accuracies[0], color=color_acc, linestyle='--', alpha=0.4, label=f'Baseline (n=1): {accuracies[0]:.0f}%')

ax2 = ax1.twinx()
color_cost = '#e74c3c'
costs = [n * 100 for n in n_values]
ax2.bar([x + 0.2 for x in n_values], costs, width=0.35, alpha=0.3, color=color_cost, label='Relative Cost')
ax2.set_ylabel('Relative Token Cost (%)', fontsize=12, color=color_cost)
ax2.tick_params(axis='y', labelcolor=color_cost)

for n, acc in zip(n_values, accuracies):
    ax1.annotate(f'{acc:.0f}%', (n, acc), textcoords='offset points', xytext=(0, 12), ha='center', fontsize=11, fontweight='bold')

ax1.set_title('Self-Consistency: Accuracy vs. Cost Trade-Off', fontsize=14)
ax1.legend(loc='upper left')
plt.tight_layout()
plt.show()

### Self-Consistency Analysis

**Q1: At what n does accuracy plateau?** TODO

**Q2: Best accuracy gain per additional cost?** TODO

**Q3: Problems where self-consistency didn't help? Why?** TODO

**Q4: When to recommend self-consistency in production?** TODO

---
# WRAP-UP

## Technique Selection Guide

Write a 1-paragraph guide: for a task of your choice, which techniques would you combine and why?

TODO

## Key Takeaways

1. TODO
2. TODO
3. TODO

---

## Before You Submit

1. ✅ All 3 extraction phases with compliance rates
2. ✅ Compliance comparison chart
3. ✅ Self-consistency experiment + gain-vs-cost chart
4. ✅ All TODOs replaced
5. ✅ Technique selection guide written
6. ✅ `git add -A && git commit -m 'Lab 4 complete' && git push`

---

## Portfolio Entry

Lab 4 contributes **Use Case #4: Structured Metadata Extraction** — with technique comparison table.

## Preview: Lab 5

In **Lab 5 — Test-Driven Prompting**, you'll apply these techniques to code generation using pytest as the external oracle.

---
*Lab 4 of 8 — DevAssist / TaskFlow Lab Series*